In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.fx.experimental.unification.unification_tools import groupby


In [67]:
df_campaigns=pd.read_csv("datasets/dim_campaigns.csv")
df_campaigns

,campaign_id,campaign_name,start_date,end_date
0,CAMP_DIW_01,Diwali,12-11-2023,18-11-2023
1,CAMP_SAN_01,Sankranti,10-01-2024,16-01-2024


In [68]:
df_campaigns.shape

(2, 4)

In [69]:
df_products=pd.read_csv("datasets/dim_products.csv")
df_products.head()

,product_code,product_name,category
0,P01,Atliq_Masoor_Dal (1KG),Grocery & Staples
1,P02,Atliq_Sonamasuri_Rice (10KG),Grocery & Staples
2,P03,Atliq_Suflower_Oil (1L),Grocery & Staples
3,P04,Atliq_Farm_Chakki_Atta (1KG),Grocery & Staples
4,P05,Atliq_Scrub_Sponge_For_Dishwash,Home Care


In [70]:
df_products.shape

(15, 3)

In [71]:
df_products.isnull().sum()

product_code    0
product_name    0
category        0
dtype: int64

In [72]:
df_stores=pd.read_csv("datasets/dim_stores.csv")
df_stores.head()

,store_id,city
0,STTRV-0,Trivandrum
1,STMDU-3,Madurai
2,STHYD-6,Hyderabad
3,STVSK-1,Visakhapatnam
4,STCBE-3,Coimbatore


In [73]:
df_stores.shape

(50, 2)

In [74]:
df_stores.isnull().sum()

store_id    0
city        0
dtype: int64

In [75]:
df_events=pd.read_csv("datasets/fact_events.csv")
df_events.head()

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622
2,f30579,STBLR-9,CAMP_DIW_01,P02,860,337.0,33% OFF,576,488
3,4f570c,STBLR-7,CAMP_DIW_01,P05,55,122.0,25% OFF,41,107
4,6d153f,STHYD-5,CAMP_SAN_01,P15,3000,122.0,500 Cashback,2500,272


In [76]:
df_events.shape

(1510, 9)

In [77]:
df_events.isnull().sum()

event_id                        0
store_id                        0
campaign_id                     0
product_code                    0
base_price(before_promo)        0
quantity_sold(before_promo)    20
promo_type                      0
base_price(after_promo)         0
quantity_sold(after_promo)      0
dtype: int64

# 1. The operations team wants to ensure the integrity of the events data by removing duplicates. Check for and remove duplicate rows in the events dataframe based on store_id, campaign_id, and product_code. How many duplicate rows were removed?

In [78]:
df_events.duplicated(["store_id","campaign_id", "product_code"]).sum()

10

In [79]:
len(df_events[df_events.duplicated(["store_id","campaign_id", "product_code"])])

10

In [80]:
df_events[df_events.duplicated(["store_id","campaign_id", "product_code"])]

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
74,46d57b,STBLR-9,CAMP_DIW_01,P13,350,94.0,BOGOF,175,329
410,f6aa36,STVSK-3,CAMP_DIW_01,P06,415,63.0,25% OFF,311,54
541,6.24E+11,STMLR-0,CAMP_SAN_01,P03,200,206.0,BOGOF,100,541
759,0f8686,STVSK-0,CAMP_SAN_01,P07,300,24.0,BOGOF,150,92
922,491ff2,STVJD-1,CAMP_SAN_01,P12,62,30.0,50% OFF,31,42
1003,4ad12b,STBLR-4,CAMP_DIW_01,P15,3000,407.0,500 Cashback,2500,1245
1155,cd4c5a,STMLR-0,CAMP_SAN_01,P09,90,34.0,25% OFF,67,28
1227,0491f4,STVSK-2,CAMP_DIW_01,P11,190,38.0,50% OFF,95,58
1313,5c7b20,STCBE-4,CAMP_DIW_01,P05,55,101.0,25% OFF,41,89
1401,d622c3,STHYD-0,CAMP_SAN_01,P10,50,25.0,25% OFF,37,20


In [81]:
df_events=df_events.drop_duplicates(["store_id","campaign_id", "product_code"])

In [82]:
df_events.duplicated(["store_id","campaign_id", "product_code"]).sum()

0

<h3> Total 10 duplicate rows are removed!</h3>

---

# 2. How many cities have more than 5 stores?

In [91]:
city_counts = df_stores.groupby("city").size()

answer = (city_counts > 5).sum()

print(answer)

3


# 3. The sales team has identified missing values in the quantity_sold(before_promo) data. Estimate these values using the median quantity sold before the promotion. How many missing values were filled, and what is the median used for imputation?


In [96]:
df_events.isnull().sum()

event_id                        0
store_id                        0
campaign_id                     0
product_code                    0
base_price(before_promo)        0
quantity_sold(before_promo)    20
promo_type                      0
base_price(after_promo)         0
quantity_sold(after_promo)      0
dtype: int64

In [99]:
fill=df_events["quantity_sold(before_promo)"].median()
fill

78.0

In [94]:
df_events["quantity_sold(before_promo)"].isnull().sum()

20

In [101]:
df_events["quantity_sold(before_promo)"] = df_events["quantity_sold(before_promo)"].fillna(fill)

In [102]:
df_events["quantity_sold(before_promo)"].isnull().sum()

0

# 4. Identify the product category with the lowest base price before the promotion

In [107]:
df_events.columns

Index(['event_id', 'store_id', 'campaign_id', 'product_code',
       'base_price(before_promo)', 'quantity_sold(before_promo)', 'promo_type',
       'base_price(after_promo)', 'quantity_sold(after_promo)'],
      dtype='str')

In [108]:
df_campaigns.columns

Index(['campaign_id', 'campaign_name', 'start_date', 'end_date'], dtype='str')

In [111]:
df_products.columns

Index(['product_code', 'product_name', 'category'], dtype='str')

In [112]:
df_stores.columns

Index(['store_id', 'city'], dtype='str')

In [113]:
df = (
    df_events
    .merge(df_campaigns, on="campaign_id", how="left")
    .merge(df_products, on="product_code", how="left")
    .merge(df_stores, on="store_id", how="left")
)

In [114]:
df.head()

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),campaign_name,start_date,end_date,product_name,category,city
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287,Diwali,12-11-2023,18-11-2023,Atliq_Farm_Chakki_Atta (1KG),Grocery & Staples,Chennai
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622,Sankranti,10-01-2024,16-01-2024,Atliq_Farm_Chakki_Atta (1KG),Grocery & Staples,Chennai
2,f30579,STBLR-9,CAMP_DIW_01,P02,860,337.0,33% OFF,576,488,Diwali,12-11-2023,18-11-2023,Atliq_Sonamasuri_Rice (10KG),Grocery & Staples,Bengaluru
3,4f570c,STBLR-7,CAMP_DIW_01,P05,55,122.0,25% OFF,41,107,Diwali,12-11-2023,18-11-2023,Atliq_Scrub_Sponge_For_Dishwash,Home Care,Bengaluru
4,6d153f,STHYD-5,CAMP_SAN_01,P15,3000,122.0,500 Cashback,2500,272,Sankranti,10-01-2024,16-01-2024,Atliq_Home_Essential_8_Product_Combo,Combo1,Hyderabad


In [124]:
df.groupby("category")["base_price(before_promo)"].mean()

category
Combo1               3000.000
Grocery & Staples     385.000
Home Appliances       685.000
Home Care             490.000
Personal Care         102.375
Name: base_price(before_promo), dtype: float64

In [119]:
df.loc[df["base_price(before_promo)"].idxmin(), "category"]

'Personal Care'

[# 5. What is the total quantity sold after the promotion for the BOGOF promo type during the Diwali campaign


In [127]:
df.groupby("promo_type")["quantity_sold(after_promo)"].mean()["BOGOF"]

430.506

In [133]:
df.groupby(["campaign_name", "promo_type"])["quantity_sold(after_promo)"].sum().loc["Diwali", "BOGOF"]

34461

In [134]:
df[
    (df["promo_type"] == "BOGOF") &
    (df["campaign_name"] == "Diwali")
]["quantity_sold(after_promo)"].sum()

34461

# Which store recorded the highest quantity sold after the promotion during the Diwali campaign?


In [135]:
df.head(2)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),campaign_name,start_date,end_date,product_name,category,city
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287,Diwali,12-11-2023,18-11-2023,Atliq_Farm_Chakki_Atta (1KG),Grocery & Staples,Chennai
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622,Sankranti,10-01-2024,16-01-2024,Atliq_Farm_Chakki_Atta (1KG),Grocery & Staples,Chennai


In [162]:
df[df["campaign_name"] == "Diwali"]\
    .groupby("store_id")["quantity_sold(after_promo)"] .sum().idxmax()

'STCHE-4'